# Honest alternative feature selection - 03 outer-fold evaluation

This notebook consumes only artifacts created by notebooks 01 and 02. It now selects global candidate `recipe_config_id` values from notebook-02 round-2 scores and audits each selected configuration on every outer fold.

The key invariant is that final `stable_outer_configurations.csv` may only contain configurations with `outer_fold_count == 5`. This avoids survivorship bias from evaluating a config only on the folds where it survived notebook-02 inner-OOF prefiltering.

For final selection this notebook uses a rate-matched and population-scaled outer score. The final submission selects top 1000 from the full test set, so each 1000-row outer fold is capped at about 200 rows. The TP/FP gross value from that fold is then scaled back to the full test population, while feature cost is charged once.


## 0. Setup


In [2]:
from __future__ import annotations

import json
import os
from datetime import UTC, datetime

import pandas as pd

from cost_effective.dataset import find_project_root, load_test_data, load_training_data
from cost_effective.dataset.utils import DEFAULT_MAX_TARGETS
from cost_effective.models.feature_selection_alternative_evaluation import (
    aggregate_outer_scores,
    complete_outer_prediction_pairs,
    filter_outer_recipe_pairs,
    rescore_outer_predictions_at_targets,
    run_outer_fold_evaluation,
    select_cost_aware_configurations,
    select_global_stage02_pipeline_recipes,
    select_stable_configurations,
)
from cost_effective.models.feature_selection_alternative_inner_selection import (
    load_stage_one_tables,
)
from cost_effective.notebook_artifacts import read_csv_if_available

project_root = find_project_root()
USE_EXISTING_OUTPUTS = True
FORCE_RERUN = False

stage_one_dir = (
    project_root / "outputs" / "feature_selection_alternative" / "01_fold_plan_and_recipe_space"
)
stage_two_dir = (
    project_root / "outputs" / "feature_selection_alternative" / "02_inner_recipe_selection"
)
outputs = project_root / "outputs" / "feature_selection_alternative" / "03_fold_evaluation"
outputs.mkdir(parents=True, exist_ok=True)

required_stage_one_files = (
    "outer_fold_assignments.csv",
    "inner_fold_assignments.csv",
    "prescreen_recipe_space.csv",
    "feature_size_grid.csv",
    "model_spec_space.csv",
    "pipeline_recipe_space.csv",
    "leakage_contract.csv",
)
missing_stage_one_files = [
    name for name in required_stage_one_files if not (stage_one_dir / name).exists()
]
if missing_stage_one_files:
    raise FileNotFoundError(
        "Missing setup artifacts for the alternative feature-selection path: "
        f"{missing_stage_one_files}. Run "
        "notebooks/alternative_approach/modeling_feature_selection_alternative_setup.ipynb first."
    )

required_stage_two_files = ("round2_inner_oof_scores.csv",)
missing_stage_two_files = [
    name for name in required_stage_two_files if not (stage_two_dir / name).exists()
]
if missing_stage_two_files:
    raise FileNotFoundError(
        "Missing inner-selection artifacts: "
        f"{missing_stage_two_files}. Run "
        "notebooks/alternative_approach/modeling_feature_selection_alternative_inner_selection.ipynb first."
    )

X_train, y_train = load_training_data(project_root / "data")
X_test_row_count = len(load_test_data(project_root / "data"))
stage_one = load_stage_one_tables(stage_one_dir)

outer_assignments = stage_one["outer_assignments"]
inner_assignments = stage_one["inner_assignments"]
prescreen_recipes = stage_one["prescreen_recipes"]
model_specs = stage_one["model_specs"]

X_train.shape, y_train.shape, outer_assignments.shape, X_test_row_count

((5000, 500), (5000,), (5000, 2), 5001)

In [3]:
RANDOM_STATE = 42
MAX_TARGETS = DEFAULT_MAX_TARGETS
OUTER_FOLDS_TO_RUN = None  # None means all outer folds declared in notebook 01.

STAGE03_GLOBAL_CONFIGS = int(os.environ.get("ALTERNATIVE_FS_EVAL_GLOBAL_CONFIGS", "20"))
STAGE03_MAX_FEATURE_SIZE = int(os.environ.get("ALTERNATIVE_FS_EVAL_MAX_FEATURE_SIZE", "6"))
MIN_INNER_FOLD_COUNT_ROUND2 = 5

RUN_OUTER_REFIT = bool(int(os.environ.get("ALTERNATIVE_FS_EVAL_RUN_REFIT", "1")))

TEST_TOP_N = DEFAULT_MAX_TARGETS
TEST_ROW_COUNT = X_test_row_count
TEST_CONTACT_RATE = min(1.0, float(TEST_TOP_N) / float(TEST_ROW_COUNT))
USE_RATE_MATCHED_OUTER_SELECTION = True
COST_AWARE_MAX_FEATURE_SIZE = STAGE03_MAX_FEATURE_SIZE
COST_AWARE_FALLBACK_MIN_OUTER_FOLDS = None
TOP_STABLE_CONFIGS = 10
MIN_OUTER_FOLDS_FOR_STABILITY = None

config = {
    "random_state": RANDOM_STATE,
    "max_targets": MAX_TARGETS,
    "outer_folds_to_run": OUTER_FOLDS_TO_RUN,
    "stage03_global_configs": STAGE03_GLOBAL_CONFIGS,
    "stage03_max_feature_size": STAGE03_MAX_FEATURE_SIZE,
    "min_inner_fold_count_round2": MIN_INNER_FOLD_COUNT_ROUND2,
    "run_outer_refit": RUN_OUTER_REFIT,
    "test_top_n": TEST_TOP_N,
    "test_row_count": TEST_ROW_COUNT,
    "test_contact_rate": TEST_CONTACT_RATE,
    "use_rate_matched_outer_selection": USE_RATE_MATCHED_OUTER_SELECTION,
    "cost_aware_max_feature_size": COST_AWARE_MAX_FEATURE_SIZE,
    "cost_aware_fallback_min_outer_folds": COST_AWARE_FALLBACK_MIN_OUTER_FOLDS,
    "top_stable_configs": TOP_STABLE_CONFIGS,
    "min_outer_folds_for_stability": MIN_OUTER_FOLDS_FOR_STABILITY,
    "scoring_formula": "10*TP - 5*FP - 200*n_features",
    "selection_contract": "Uses notebook-03 alternative outer predictions, rescored at final test contact rate.",
}
with (outputs / "stage_config.json").open("w") as file_obj:
    json.dump(config, file_obj, indent=2, default=str)

pd.Series(config)

random_state                                                                          42
max_targets                                                                         1000
outer_folds_to_run                                                                  None
stage03_global_configs                                                                20
stage03_max_feature_size                                                               6
min_inner_fold_count_round2                                                            5
run_outer_refit                                                                     True
test_top_n                                                                          1000
test_row_count                                                                      5001
test_contact_rate                                                                0.19996
use_rate_matched_outer_selection                                                    True
cost_aware_max_featur

In [4]:
OUTPUT_FILES = {
    "run_log": outputs / "outer_evaluation_run_log.csv",
    "stage02_selected_used": outputs / "stage02_selected_pipeline_recipes_used.csv",
    "outer_scores": outputs / "outer_fold_scores.csv",
    "outer_predictions": outputs / "outer_fold_predictions.csv",
    "outer_selected_features": outputs / "outer_fold_selected_features.csv",
    "outer_scores_rate_matched": outputs / "outer_fold_scores_rate_matched.csv",
    "outer_config_summary_full_cap": outputs / "outer_config_summary_full_cap.csv",
    "outer_config_summary": outputs / "outer_config_summary.csv",
    "stable_configs_full_cap": outputs / "stable_outer_configurations_full_cap.csv",
    "stable_configs": outputs / "stable_outer_configurations.csv",
    "feature_frequency": outputs / "outer_feature_frequency.csv",
    "manifest": outputs / "output_manifest.csv",
    "status": outputs / "stage_status_summary.json",
}


def log_event(stage: str, message: str, **payload) -> None:
    row = {
        "timestamp_utc": datetime.now(UTC).isoformat(),
        "stage": stage,
        "message": message,
        **payload,
    }
    frame = pd.DataFrame([row])
    frame.to_csv(
        OUTPUT_FILES["run_log"],
        mode="a",
        index=False,
        header=not OUTPUT_FILES["run_log"].exists(),
    )
    print(f"[{row['timestamp_utc']}] {stage}: {message} {payload}")


log_event("setup", "initialized notebook 03", output_dir=str(outputs.relative_to(project_root)))

[2026-06-08T10:16:10.182012+00:00] setup: initialized notebook 03 {'output_dir': 'outputs/feature_selection_alternative/03_fold_evaluation'}


## 1. Build global notebook-03 candidate configs from notebook-02 scores

The input is `round2_inner_oof_scores.csv` from notebook 02. Notebook 03 selects global `recipe_config_id` values once, then expands each selected config across all outer folds. This is intentionally different from consuming `outer_selected_pipeline_recipes.csv`, because that file is fold-local and can create survivorship bias.


In [5]:
outer_folds = sorted(outer_assignments["outer_fold"].drop_duplicates().astype(int).tolist())
if OUTER_FOLDS_TO_RUN is not None:
    outer_folds = [fold for fold in outer_folds if fold in set(OUTER_FOLDS_TO_RUN)]

REQUIRED_OUTER_FOLD_COUNT = len(outer_folds)
MIN_OUTER_FOLDS_FOR_STABILITY = REQUIRED_OUTER_FOLD_COUNT
COST_AWARE_FALLBACK_MIN_OUTER_FOLDS = REQUIRED_OUTER_FOLD_COUNT
config["required_outer_fold_count"] = REQUIRED_OUTER_FOLD_COUNT
config["min_outer_folds_for_stability"] = MIN_OUTER_FOLDS_FOR_STABILITY
config["cost_aware_fallback_min_outer_folds"] = COST_AWARE_FALLBACK_MIN_OUTER_FOLDS
with (outputs / "stage_config.json").open("w", encoding="utf-8") as handle:
    json.dump(config, handle, indent=2)

selected_pipeline_recipes = select_global_stage02_pipeline_recipes(
    stage_two_dir,
    outer_folds=outer_folds,
    top_n=STAGE03_GLOBAL_CONFIGS,
    min_inner_fold_count=MIN_INNER_FOLD_COUNT_ROUND2,
    max_feature_size=STAGE03_MAX_FEATURE_SIZE,
)
selected_pipeline_recipes.to_csv(OUTPUT_FILES["stage02_selected_used"], index=False)

log_event(
    "stage02_selection",
    "built global notebook-03 candidate configs from notebook-02 round2 scores",
    rows=len(selected_pipeline_recipes),
    global_config_count=int(selected_pipeline_recipes["recipe_config_id"].nunique())
    if not selected_pipeline_recipes.empty
    else 0,
    required_outer_fold_count=REQUIRED_OUTER_FOLD_COUNT,
    max_feature_size=STAGE03_MAX_FEATURE_SIZE,
    outer_folds=outer_folds,
)
selected_pipeline_recipes.head(30)

[2026-06-08T10:16:15.615265+00:00] stage02_selection: built global notebook-03 candidate configs from notebook-02 round2 scores {'rows': 100, 'global_config_count': 20, 'required_outer_fold_count': 5, 'max_feature_size': 6, 'outer_folds': [1, 2, 3, 4, 5]}


,recipe_config_id,pipeline_recipe_id,feature_recipe_id,prescreen_recipe_id,prescreen_name,prescreen_methods,rank_aggregation,feature_size,model_spec_id,base_model_family,...,median_inner_oof_business_score,min_inner_oof_business_score,max_inner_oof_business_score,inner_oof_optimal_k,inner_oof_f1_score,inner_oof_roc_auc_score,global_candidate_rank,inner_selection_rank,selection_artifact,outer_fold
0,ps_mean_0032__k_005__extra_trees_small_deep02,ps_mean_0032__k_005__extra_trees_small_deep02,ps_mean_0032__k_005,ps_mean_0032,pair__lightgbm_gain+sparse_gam_spam,"[""lightgbm_gain"", ""sparse_gam_spam""]",mean_rank_score,5,extra_trees_small_deep02,extra_trees_small,...,4985.0,4985.0,4985.0,999.0,0.489796,0.682412,1,1,round2_inner_oof_scores.csv,1
1,ps_mean_0007__k_006__extra_trees_small_deep01,ps_mean_0007__k_006__extra_trees_small_deep01,ps_mean_0007__k_006,ps_mean_0007,single__sparse_gam_spam,"[""sparse_gam_spam""]",mean_rank_score,6,extra_trees_small_deep01,extra_trees_small,...,5000.0,4860.0,5030.0,999.0,0.498118,0.691210,2,2,round2_inner_oof_scores.csv,1
2,ps_mean_0007__k_006__extra_trees_small_deep02,ps_mean_0007__k_006__extra_trees_small_deep02,ps_mean_0007__k_006,ps_mean_0007,single__sparse_gam_spam,"[""sparse_gam_spam""]",mean_rank_score,6,extra_trees_small_deep02,extra_trees_small,...,4960.0,4875.0,5070.0,999.5,0.497867,0.693830,3,3,round2_inner_oof_scores.csv,1
3,ps_mean_0052__k_006__extra_trees_small_deep01,ps_mean_0052__k_006__extra_trees_small_deep01,ps_mean_0052__k_006,ps_mean_0052,triple__mutual_info+lightgbm_gain+ebm,"[""mutual_info"", ""lightgbm_gain"", ""ebm""]",mean_rank_score,6,extra_trees_small_deep01,extra_trees_small,...,4960.0,4960.0,4960.0,1000.0,0.497492,0.678581,4,4,round2_inner_oof_scores.csv,1
4,ps_mean_0032__k_005__extra_trees_small_deep03,ps_mean_0032__k_005__extra_trees_small_deep03,ps_mean_0032__k_005,ps_mean_0032,pair__lightgbm_gain+sparse_gam_spam,"[""lightgbm_gain"", ""sparse_gam_spam""]",mean_rank_score,5,extra_trees_small_deep03,extra_trees_small,...,4950.0,4950.0,4950.0,1000.0,0.488294,0.683566,5,5,round2_inner_oof_scores.csv,1
5,ps_mean_0052__k_006__xgboost_classifier_deep04,ps_mean_0052__k_006__xgboost_classifier_deep04,ps_mean_0052__k_006,ps_mean_0052,triple__mutual_info+lightgbm_gain+ebm,"[""mutual_info"", ""lightgbm_gain"", ""ebm""]",mean_rank_score,6,xgboost_classifier_deep04,xgboost_classifier,...,4950.0,4950.0,4950.0,996.0,0.496824,0.688411,6,6,round2_inner_oof_scores.csv,1
6,ps_mean_0007__k_006__extra_trees_small_deep03,ps_mean_0007__k_006__extra_trees_small_deep03,ps_mean_0007__k_006,ps_mean_0007,single__sparse_gam_spam,"[""sparse_gam_spam""]",mean_rank_score,6,extra_trees_small_deep03,extra_trees_small,...,4932.5,4890.0,5020.0,996.5,0.496823,0.694644,7,7,round2_inner_oof_scores.csv,1
7,ps_mean_0007__k_006__extra_trees_small_deep08,ps_mean_0007__k_006__extra_trees_small_deep08,ps_mean_0007__k_006,ps_mean_0007,single__sparse_gam_spam,"[""sparse_gam_spam""]",mean_rank_score,6,extra_trees_small_deep08,extra_trees_small,...,5000.0,4750.0,5010.0,999.5,0.496697,0.693616,8,8,round2_inner_oof_scores.csv,1
8,ps_mean_0032__k_005__extra_trees_small_deep05,ps_mean_0032__k_005__extra_trees_small_deep05,ps_mean_0032__k_005,ps_mean_0032,pair__lightgbm_gain+sparse_gam_spam,"[""lightgbm_gain"", ""sparse_gam_spam""]",mean_rank_score,5,extra_trees_small_deep05,extra_trees_small,...,4935.0,4935.0,4935.0,1000.0,0.487625,0.682358,9,9,round2_inner_oof_scores.csv,1
9,ps_mean_0005__k_006__extra_trees_small_deep03,ps_mean_0005__k_006__extra_trees_small_deep03,ps_mean_0005__k_006,ps_mean_0005,single__lightgbm_gain,"[""lightgbm_gain""]",mean_rank_score,6,extra_trees_small_deep03,extra_trees_small,...,4960.0,4765.0,5025.0,996.6,0.496286,0.693798,10,10,round2_inner_oof_scores.csv,1


## 2. Honest outer-fold refit and evaluation

Each selected global config must have one completed audit row for every outer fold. Existing complete `(outer_fold, recipe_config_id)` pairs from `outer_fold_scores.csv` and `outer_fold_predictions.csv` are reused. Missing pairs are refitted from scratch on that fold's `outer_train` and scored on untouched `outer_val`.


In [6]:
def pair_set(frame: pd.DataFrame) -> set[tuple[int, str]]:
    if frame.empty or not {"outer_fold", "recipe_config_id"}.issubset(frame.columns):
        return set()
    return {
        (int(row.outer_fold), str(row.recipe_config_id))
        for row in frame[["outer_fold", "recipe_config_id"]]
        .drop_duplicates()
        .itertuples(index=False)
    }


target_pairs = pair_set(selected_pipeline_recipes)
existing_outer_scores = read_csv_if_available(OUTPUT_FILES["outer_scores"])
existing_outer_predictions = read_csv_if_available(OUTPUT_FILES["outer_predictions"])
existing_outer_selected_features = read_csv_if_available(OUTPUT_FILES["outer_selected_features"])
complete_existing_pairs = complete_outer_prediction_pairs(
    existing_outer_scores,
    existing_outer_predictions,
    outer_assignments,
    outer_folds=outer_folds,
)
reusable_pairs = target_pairs.intersection(complete_existing_pairs)
missing_pairs = target_pairs.difference(reusable_pairs)

selected_missing_recipes = selected_pipeline_recipes.loc[
    selected_pipeline_recipes.apply(
        lambda row: (int(row["outer_fold"]), str(row["recipe_config_id"])) in missing_pairs,
        axis=1,
    )
].copy()

log_event(
    "outer_refit_plan",
    "planned incremental complete outer audit",
    target_pairs=len(target_pairs),
    reusable_pairs=len(reusable_pairs),
    missing_pairs=len(missing_pairs),
    selected_missing_rows=len(selected_missing_recipes),
)

if RUN_OUTER_REFIT and not selected_missing_recipes.empty:
    new_outer_scores, new_outer_predictions, new_outer_selected_features = (
        run_outer_fold_evaluation(
            X_train,
            y_train,
            selected_missing_recipes,
            outer_assignments,
            prescreen_recipes,
            model_specs,
            outer_folds=outer_folds,
            random_state=RANDOM_STATE,
            max_targets=MAX_TARGETS,
        )
    )
else:
    new_outer_scores = pd.DataFrame()
    new_outer_predictions = pd.DataFrame()
    new_outer_selected_features = pd.DataFrame()
    if selected_missing_recipes.empty:
        log_event("outer_refit_skip", "all target pairs are already complete")
    elif not RUN_OUTER_REFIT:
        raise RuntimeError(
            "RUN_OUTER_REFIT=False but selected global configs have missing outer-fold audits. "
            "Set RUN_OUTER_REFIT=True to fit missing pairs."
        )

kept_existing_scores = filter_outer_recipe_pairs(existing_outer_scores, reusable_pairs)
kept_existing_predictions = filter_outer_recipe_pairs(existing_outer_predictions, reusable_pairs)
kept_existing_features = filter_outer_recipe_pairs(existing_outer_selected_features, reusable_pairs)

outer_fold_scores = pd.concat([kept_existing_scores, new_outer_scores], ignore_index=True)
outer_fold_predictions = pd.concat(
    [kept_existing_predictions, new_outer_predictions], ignore_index=True
)
outer_fold_selected_features = pd.concat(
    [kept_existing_features, new_outer_selected_features],
    ignore_index=True,
)

complete_after_refit = complete_outer_prediction_pairs(
    outer_fold_scores,
    outer_fold_predictions,
    outer_assignments,
    outer_folds=outer_folds,
)
still_missing_pairs = target_pairs.difference(complete_after_refit)
if still_missing_pairs:
    examples = sorted(still_missing_pairs)[:10]
    raise RuntimeError(f"Incomplete outer audit remains after refit; examples={examples}")

outer_fold_scores.to_csv(OUTPUT_FILES["outer_scores"], index=False)
outer_fold_predictions.to_csv(OUTPUT_FILES["outer_predictions"], index=False)
outer_fold_selected_features.to_csv(OUTPUT_FILES["outer_selected_features"], index=False)

log_event(
    "outer_refit",
    "saved complete alternative outer-fold evaluation frames",
    run_outer_refit=RUN_OUTER_REFIT,
    score_rows=len(outer_fold_scores),
    prediction_rows=len(outer_fold_predictions),
    selected_feature_rows=len(outer_fold_selected_features),
    complete_pairs=len(complete_after_refit),
    required_pairs=len(target_pairs),
    ok_scores=int(outer_fold_scores["status"].eq("ok").sum()) if not outer_fold_scores.empty else 0,
)
outer_fold_scores.head(30)

[2026-06-08T10:16:30.607032+00:00] outer_refit_plan: planned incremental complete outer audit {'target_pairs': 100, 'reusable_pairs': 0, 'missing_pairs': 100, 'selected_missing_rows': 100}
[outer=1] fitting 4 prescreen methods on 4000 outer-train rows


2026-06-08 10:16:57,361 - interpret.utils._native - INFO - EBM lib loading.
2026-06-08 10:16:57,361 - interpret.utils._native - INFO - Finding library for Linux, x86_64, bitsize=64, debug=False
2026-06-08 10:16:57,384 - interpret.utils._native - INFO - Loading EBM library /workspaces/cost-effective/.venv/lib/python3.12/site-packages/interpret/utils/../root/bld/lib/libebm_linux_x64.so
2026-06-08 10:17:03,666 - interpret.utils._compressed_dataset - INFO - Creating native dataset
2026-06-08 10:17:03,810 - interpret.glassbox._ebm._boost - INFO - Start boosting
2026-06-08 10:17:03,811 - interpret.utils._native - INFO - Booster allocation start
2026-06-08 10:17:03,839 - interpret.utils._native - INFO - Booster allocation end
2026-06-08 10:17:08,038 - interpret.utils._native - INFO - Deallocation boosting start
2026-06-08 10:17:08,041 - interpret.utils._native - INFO - Deallocation boosting end
2026-06-08 10:17:08,042 - interpret.glassbox._ebm._boost - INFO - Start boosting
2026-06-08 10:17:0

[outer=1] ps_mean_0032__k_005__extra_trees_small_deep02 score=1530.0 k=982
[outer=1] ps_mean_0007__k_006__extra_trees_small_deep01 score=1345.0 k=916
[outer=1] ps_mean_0007__k_006__extra_trees_small_deep02 score=1340.0 k=974
[outer=1] ps_mean_0052__k_006__extra_trees_small_deep01 score=1430.0 k=722
[outer=1] ps_mean_0032__k_005__extra_trees_small_deep03 score=1530.0 k=985
[outer=1] ps_mean_0052__k_006__xgboost_classifier_deep04 score=1495.0 k=832
[outer=1] ps_mean_0007__k_006__extra_trees_small_deep03 score=1335.0 k=927
[outer=1] ps_mean_0007__k_006__extra_trees_small_deep08 score=1350.0 k=927
[outer=1] ps_mean_0032__k_005__extra_trees_small_deep05 score=1515.0 k=979
[outer=1] ps_mean_0005__k_006__extra_trees_small_deep03 score=1395.0 k=888
[outer=1] ps_mean_0007__k_006__extra_trees_small_deep04 score=1335.0 k=927
[outer=1] ps_mean_0005__k_006__extra_trees_small_deep02 score=1375.0 k=892
[outer=1] ps_mean_0052__k_006__xgboost_classifier_deep03 score=1505.0 k=707
[outer=1] ps_mean_0052_

2026-06-08 10:18:49,624 - interpret.utils._compressed_dataset - INFO - Creating native dataset
2026-06-08 10:18:49,857 - interpret.glassbox._ebm._boost - INFO - Start boosting
2026-06-08 10:18:49,858 - interpret.utils._native - INFO - Booster allocation start
2026-06-08 10:18:49,882 - interpret.utils._native - INFO - Booster allocation end
2026-06-08 10:18:54,820 - interpret.utils._native - INFO - Deallocation boosting start
2026-06-08 10:18:54,822 - interpret.utils._native - INFO - Deallocation boosting end
2026-06-08 10:18:54,823 - interpret.glassbox._ebm._boost - INFO - Start boosting
2026-06-08 10:18:54,823 - interpret.utils._native - INFO - Booster allocation start
2026-06-08 10:18:54,848 - interpret.utils._native - INFO - Booster allocation end
2026-06-08 10:18:58,819 - interpret.utils._native - INFO - Deallocation boosting start
2026-06-08 10:18:58,825 - interpret.utils._native - INFO - Deallocation boosting end
2026-06-08 10:18:58,826 - interpret.glassbox._ebm._boost - INFO - S

[outer=2] ps_mean_0032__k_005__extra_trees_small_deep02 score=1540.0 k=929
[outer=2] ps_mean_0007__k_006__extra_trees_small_deep01 score=1425.0 k=822
[outer=2] ps_mean_0007__k_006__extra_trees_small_deep02 score=1430.0 k=845
[outer=2] ps_mean_0052__k_006__extra_trees_small_deep01 score=1360.0 k=739
[outer=2] ps_mean_0032__k_005__extra_trees_small_deep03 score=1560.0 k=859
[outer=2] ps_mean_0052__k_006__xgboost_classifier_deep04 score=1375.0 k=817
[outer=2] ps_mean_0007__k_006__extra_trees_small_deep03 score=1465.0 k=814
[outer=2] ps_mean_0007__k_006__extra_trees_small_deep08 score=1450.0 k=853
[outer=2] ps_mean_0032__k_005__extra_trees_small_deep05 score=1555.0 k=923
[outer=2] ps_mean_0005__k_006__extra_trees_small_deep03 score=1490.0 k=764
[outer=2] ps_mean_0007__k_006__extra_trees_small_deep04 score=1445.0 k=866
[outer=2] ps_mean_0005__k_006__extra_trees_small_deep02 score=1455.0 k=756
[outer=2] ps_mean_0052__k_006__xgboost_classifier_deep03 score=1345.0 k=691
[outer=2] ps_mean_0052_

...

2026-06-08 10:21:52,417 - interpret.utils._compressed_dataset - INFO - Creating native dataset
2026-06-08 10:21:52,551 - interpret.glassbox._ebm._boost - INFO - Start boosting
2026-06-08 10:21:52,551 - interpret.utils._native - INFO - Booster allocation start
2026-06-08 10:21:52,572 - interpret.utils._native - INFO - Booster allocation end
2026-06-08 10:21:56,770 - interpret.utils._native - INFO - Deallocation boosting start
2026-06-08 10:21:56,771 - interpret.utils._native - INFO - Deallocation boosting end
2026-06-08 10:21:56,772 - interpret.glassbox._ebm._boost - INFO - Start boosting
2026-06-08 10:21:56,772 - interpret.utils._native - INFO - Booster allocation start
2026-06-08 10:21:56,799 - interpret.utils._native - INFO - Booster allocation end
2026-06-08 10:22:00,894 - interpret.utils._native - INFO - Deallocation boosting start
2026-06-08 10:22:00,895 - interpret.utils._native - INFO - Deallocation boosting end
2026-06-08 10:22:00,897 - interpret.glassbox._ebm._boost - INFO - S

[outer=4] ps_mean_0032__k_005__extra_trees_small_deep02 score=1710.0 k=811
[outer=4] ps_mean_0007__k_006__extra_trees_small_deep01 score=1515.0 k=831
[outer=4] ps_mean_0007__k_006__extra_trees_small_deep02 score=1520.0 k=851
[outer=4] ps_mean_0052__k_006__extra_trees_small_deep01 score=1305.0 k=966
[outer=4] ps_mean_0032__k_005__extra_trees_small_deep03 score=1685.0 k=771
[outer=4] ps_mean_0052__k_006__xgboost_classifier_deep04 score=1280.0 k=995
[outer=4] ps_mean_0007__k_006__extra_trees_small_deep03 score=1530.0 k=852
[outer=4] ps_mean_0007__k_006__extra_trees_small_deep08 score=1525.0 k=832
[outer=4] ps_mean_0032__k_005__extra_trees_small_deep05 score=1700.0 k=795
[outer=4] ps_mean_0005__k_006__extra_trees_small_deep03 score=1515.0 k=768
[outer=4] ps_mean_0007__k_006__extra_trees_small_deep04 score=1535.0 k=776
[outer=4] ps_mean_0005__k_006__extra_trees_small_deep02 score=1500.0 k=753
[outer=4] ps_mean_0052__k_006__xgboost_classifier_deep03 score=1275.0 k=987
[outer=4] ps_mean_0052_

2026-06-08 10:23:24,371 - interpret.utils._compressed_dataset - INFO - Creating native dataset
2026-06-08 10:23:24,515 - interpret.glassbox._ebm._boost - INFO - Start boosting
2026-06-08 10:23:24,515 - interpret.utils._native - INFO - Booster allocation start
2026-06-08 10:23:24,534 - interpret.utils._native - INFO - Booster allocation end
2026-06-08 10:23:28,482 - interpret.utils._native - INFO - Deallocation boosting start
2026-06-08 10:23:28,483 - interpret.utils._native - INFO - Deallocation boosting end
2026-06-08 10:23:28,484 - interpret.glassbox._ebm._boost - INFO - Start boosting
2026-06-08 10:23:28,485 - interpret.utils._native - INFO - Booster allocation start
2026-06-08 10:23:28,511 - interpret.utils._native - INFO - Booster allocation end
2026-06-08 10:23:32,268 - interpret.utils._native - INFO - Deallocation boosting start
2026-06-08 10:23:32,270 - interpret.utils._native - INFO - Deallocation boosting end
2026-06-08 10:23:32,271 - interpret.glassbox._ebm._boost - INFO - S

[outer=5] ps_mean_0032__k_005__extra_trees_small_deep02 score=1645.0 k=617
[outer=5] ps_mean_0007__k_006__extra_trees_small_deep01 score=1540.0 k=709
[outer=5] ps_mean_0007__k_006__extra_trees_small_deep02 score=1560.0 k=690
[outer=5] ps_mean_0052__k_006__extra_trees_small_deep01 score=1435.0 k=766
[outer=5] ps_mean_0032__k_005__extra_trees_small_deep03 score=1635.0 k=685
[outer=5] ps_mean_0052__k_006__xgboost_classifier_deep04 score=1395.0 k=810
[outer=5] ps_mean_0007__k_006__extra_trees_small_deep03 score=1565.0 k=731
[outer=5] ps_mean_0007__k_006__extra_trees_small_deep08 score=1560.0 k=735
[outer=5] ps_mean_0032__k_005__extra_trees_small_deep05 score=1635.0 k=619
[outer=5] ps_mean_0005__k_006__extra_trees_small_deep03 score=1530.0 k=870
[outer=5] ps_mean_0007__k_006__extra_trees_small_deep04 score=1555.0 k=703
[outer=5] ps_mean_0005__k_006__extra_trees_small_deep02 score=1530.0 k=864
[outer=5] ps_mean_0052__k_006__xgboost_classifier_deep03 score=1425.0 k=882
[outer=5] ps_mean_0052_

,recipe_config_id,pipeline_recipe_id,feature_recipe_id,prescreen_recipe_id,prescreen_name,prescreen_methods,rank_aggregation,feature_size,model_spec_id,base_model_family,...,outer_optimal_k,outer_tp_at_optimal_k,outer_fp_at_optimal_k,outer_gross_score,feature_penalty,outer_f1_score,outer_roc_auc_score,outer_n_predictions,outer_selected_rate,outer_threshold
0,ps_mean_0032__k_005__extra_trees_small_deep08,ps_mean_0032__k_005__extra_trees_small_deep08,ps_mean_0032__k_005,ps_mean_0032,pair__lightgbm_gain+sparse_gam_spam,"[""lightgbm_gain"", ""sparse_gam_spam""]",mean_rank_score,5,extra_trees_small_deep08,extra_trees_small,...,982,497,485,2545.0,1000.0,0.672076,0.656424,1000,0.982,0.446135
1,ps_mean_0032__k_005__extra_trees_small_deep02,ps_mean_0032__k_005__extra_trees_small_deep02,ps_mean_0032__k_005,ps_mean_0032,pair__lightgbm_gain+sparse_gam_spam,"[""lightgbm_gain"", ""sparse_gam_spam""]",mean_rank_score,5,extra_trees_small_deep02,extra_trees_small,...,982,496,486,2530.0,1000.0,0.670723,0.654364,1000,0.982,0.457150
2,ps_mean_0032__k_005__extra_trees_small_deep03,ps_mean_0032__k_005__extra_trees_small_deep03,ps_mean_0032__k_005,ps_mean_0032,pair__lightgbm_gain+sparse_gam_spam,"[""lightgbm_gain"", ""sparse_gam_spam""]",mean_rank_score,5,extra_trees_small_deep03,extra_trees_small,...,985,497,488,2530.0,1000.0,0.670715,0.655848,1000,0.985,0.447897
3,ps_mean_0032__k_005__extra_trees_small_deep06,ps_mean_0032__k_005__extra_trees_small_deep06,ps_mean_0032__k_005,ps_mean_0032,pair__lightgbm_gain+sparse_gam_spam,"[""lightgbm_gain"", ""sparse_gam_spam""]",mean_rank_score,5,extra_trees_small_deep06,extra_trees_small,...,982,496,486,2530.0,1000.0,0.670723,0.657236,1000,0.982,0.436086
4,ps_mean_0032__k_005__extra_trees_small_deep05,ps_mean_0032__k_005__extra_trees_small_deep05,ps_mean_0032__k_005,ps_mean_0032,pair__lightgbm_gain+sparse_gam_spam,"[""lightgbm_gain"", ""sparse_gam_spam""]",mean_rank_score,5,extra_trees_small_deep05,extra_trees_small,...,979,494,485,2515.0,1000.0,0.669377,0.656344,1000,0.979,0.438461
5,ps_mean_0032__k_005__extra_trees_small_deep01,ps_mean_0032__k_005__extra_trees_small_deep01,ps_mean_0032__k_005,ps_mean_0032,pair__lightgbm_gain+sparse_gam_spam,"[""lightgbm_gain"", ""sparse_gam_spam""]",mean_rank_score,5,extra_trees_small_deep01,extra_trees_small,...,980,494,486,2510.0,1000.0,0.668923,0.655392,1000,0.980,0.465916
6,ps_mean_0052__k_006__xgboost_classifier_deep03,ps_mean_0052__k_006__xgboost_classifier_deep03,ps_mean_0052__k_006,ps_mean_0052,triple__mutual_info+lightgbm_gain+ebm,"[""mutual_info"", ""lightgbm_gain"", ""ebm""]",mean_rank_score,6,xgboost_classifier_deep03,xgboost_classifier,...,707,416,291,2705.0,1200.0,0.691030,0.695553,1000,0.707,0.370827
7,ps_mean_0052__k_006__xgboost_classifier_deep04,ps_mean_0052__k_006__xgboost_classifier_deep04,ps_mean_0052__k_006,ps_mean_0052,triple__mutual_info+lightgbm_gain+ebm,"[""mutual_info"", ""lightgbm_gain"", ""ebm""]",mean_rank_score,6,xgboost_classifier_deep04,xgboost_classifier,...,832,457,375,2695.0,1200.0,0.688312,0.694745,1000,0.832,0.354969
8,ps_mean_0052__k_006__xgboost_classifier_deep06,ps_mean_0052__k_006__xgboost_classifier_deep06,ps_mean_0052__k_006,ps_mean_0052,triple__mutual_info+lightgbm_gain+ebm,"[""mutual_info"", ""lightgbm_gain"", ""ebm""]",mean_rank_score,6,xgboost_classifier_deep06,xgboost_classifier,...,759,432,327,2685.0,1200.0,0.687898,0.693373,1000,0.759,0.422549
9,ps_mean_0052__k_006__extra_trees_small_deep08,ps_mean_0052__k_006__extra_trees_small_deep08,ps_mean_0052__k_006,ps_mean_0052,triple__mutual_info+lightgbm_gain+ebm,"[""mutual_info"", ""lightgbm_gain"", ""ebm""]",mean_rank_score,6,extra_trees_small_deep08,extra_trees_small,...,805,446,359,2665.0,1200.0,0.685811,0.698469,1000,0.805,0.472497


## 3. Rate-matched, population-scaled finalist selection

The raw outer score allows `k <= 1000` on a 1000-row outer fold, which effectively permits contacting the entire fold. The final test ranking selects 1000 from about 5000 rows, so finalist selection below caps each outer fold at the same contact rate, around 200 rows.

Because one outer fold is only about one fifth of the final test population, the TP/FP gross value is scaled back to the full test row count before subtracting feature cost once. The main `stable_outer_configurations.csv` is selected from this scaled score, with a cost-aware feature-size preference. The old full-cap summary is still saved for diagnostics.


In [ ]:
outer_val_row_counts = (
    outer_assignments.loc[outer_assignments["outer_fold"].isin(outer_folds)]
    .groupby("outer_fold")["sample_index"]
    .size()
    .to_dict()
)
rate_matched_max_targets_by_outer = {
    int(outer_fold): max(1, min(int(row_count), round(row_count * TEST_CONTACT_RATE)))
    for outer_fold, row_count in outer_val_row_counts.items()
}
gross_score_scale_by_outer = {
    int(outer_fold): float(TEST_ROW_COUNT) / float(row_count)
    for outer_fold, row_count in outer_val_row_counts.items()
}

outer_fold_scores_rate_matched = rescore_outer_predictions_at_targets(
    outer_fold_predictions,
    max_targets_by_outer_fold=rate_matched_max_targets_by_outer,
    gross_score_scale_by_outer_fold=gross_score_scale_by_outer,
    original_scores=outer_fold_scores,
    scoring_strategy="rate_matched_test_top1000_contact_rate_scaled_gross",
)
outer_fold_scores_rate_matched.to_csv(OUTPUT_FILES["outer_scores_rate_matched"], index=False)

outer_config_summary_full_cap = aggregate_outer_scores(outer_fold_scores)
outer_config_summary = aggregate_outer_scores(outer_fold_scores_rate_matched)

stable_outer_configurations_full_cap = select_stable_configurations(
    outer_config_summary_full_cap.loc[
        outer_config_summary_full_cap["outer_fold_count"].eq(REQUIRED_OUTER_FOLD_COUNT)
    ].copy(),
    top_n=TOP_STABLE_CONFIGS,
    min_outer_folds=REQUIRED_OUTER_FOLD_COUNT,
)
stable_outer_configurations = select_cost_aware_configurations(
    outer_config_summary,
    top_n=TOP_STABLE_CONFIGS,
    min_outer_folds=REQUIRED_OUTER_FOLD_COUNT,
    max_feature_size=COST_AWARE_MAX_FEATURE_SIZE,
    fallback_min_outer_folds=REQUIRED_OUTER_FOLD_COUNT,
    require_exact_outer_folds=True,
    allow_incomplete_fallback=False,
)
if (
    not stable_outer_configurations.empty
    and not stable_outer_configurations["outer_fold_count"].eq(REQUIRED_OUTER_FOLD_COUNT).all()
):
    raise RuntimeError("Final stable configs include incomplete outer-fold audits")

outer_config_summary_full_cap.to_csv(OUTPUT_FILES["outer_config_summary_full_cap"], index=False)
outer_config_summary.to_csv(OUTPUT_FILES["outer_config_summary"], index=False)
stable_outer_configurations_full_cap.to_csv(OUTPUT_FILES["stable_configs_full_cap"], index=False)
stable_outer_configurations.to_csv(OUTPUT_FILES["stable_configs"], index=False)

log_event(
    "aggregation",
    "saved rate-matched population-scaled cost-aware finalists",
    contact_rate=TEST_CONTACT_RATE,
    max_targets_by_outer=rate_matched_max_targets_by_outer,
    gross_score_scale_by_outer=gross_score_scale_by_outer,
    full_cap_config_rows=len(outer_config_summary_full_cap),
    rate_matched_config_rows=len(outer_config_summary),
    finalist_rows=len(stable_outer_configurations),
    required_outer_fold_count=REQUIRED_OUTER_FOLD_COUNT,
    cost_aware_max_feature_size=COST_AWARE_MAX_FEATURE_SIZE,
)
stable_outer_configurations

In [ ]:
if not outer_fold_selected_features.empty:
    outer_feature_frequency = (
        outer_fold_selected_features.loc[outer_fold_selected_features["status"].eq("ok")]
        .groupby("feature", as_index=False)
        .agg(
            count=("feature", "size"),
            outer_fold_count=("outer_fold", "nunique"),
            mean_feature_order=("feature_order", "mean"),
            recipe_config_count=("recipe_config_id", "nunique"),
        )
        .sort_values(
            ["count", "outer_fold_count", "mean_feature_order"], ascending=[False, False, True]
        )
        .reset_index(drop=True)
    )
else:
    outer_feature_frequency = pd.DataFrame()

outer_feature_frequency.to_csv(OUTPUT_FILES["feature_frequency"], index=False)
outer_feature_frequency.head(50)

## 4. Output manifest


In [ ]:
status_summary = {
    "scoring_version": "alternative_feature_selection_rate_matched_topk_v3_scaled_gross",
    "selection_strategy": "rate_matched_contact_rate_scaled_gross_cost_aware",
    "test_top_n": TEST_TOP_N,
    "test_row_count": TEST_ROW_COUNT,
    "test_contact_rate": TEST_CONTACT_RATE,
    "rate_matched_max_targets_by_outer": rate_matched_max_targets_by_outer,
    "gross_score_scale_by_outer": gross_score_scale_by_outer,
    "cost_aware_max_feature_size": COST_AWARE_MAX_FEATURE_SIZE,
    "required_outer_fold_count": REQUIRED_OUTER_FOLD_COUNT,
    "run_outer_refit": RUN_OUTER_REFIT,
    "score_rows_full_cap": len(outer_fold_scores),
    "score_rows_rate_matched": len(outer_fold_scores_rate_matched),
    "ok_score_rows_rate_matched": int(outer_fold_scores_rate_matched["status"].eq("ok").sum())
    if not outer_fold_scores_rate_matched.empty
    else 0,
    "prediction_rows": len(outer_fold_predictions),
    "selected_feature_rows": len(outer_fold_selected_features),
    "outer_config_rows_full_cap": len(outer_config_summary_full_cap),
    "outer_config_rows_rate_matched": len(outer_config_summary),
    "stable_config_rows": len(stable_outer_configurations),
    "best_mean_outer_business_score_rate_matched": float(
        outer_config_summary.iloc[0]["mean_outer_business_score"]
    )
    if not outer_config_summary.empty
    else None,
    "best_recipe_config_id_rate_matched": str(outer_config_summary.iloc[0]["recipe_config_id"])
    if not outer_config_summary.empty
    else None,
}
with OUTPUT_FILES["status"].open("w") as file_obj:
    json.dump(status_summary, file_obj, indent=2, default=str)

output_manifest = pd.DataFrame([
    {"file": "stage_config.json", "meaning": "Notebook-03 execution config."},
    {"file": "outer_evaluation_run_log.csv", "meaning": "Timestamped notebook-03 log."},
    {
        "file": "stage02_selected_pipeline_recipes_used.csv",
        "meaning": "Notebook-02 selected recipes consumed by outer evaluation.",
    },
    {
        "file": "outer_fold_scores.csv",
        "meaning": "Original full-cap outer scores on untouched outer_val.",
    },
    {
        "file": "outer_fold_predictions.csv",
        "meaning": "Sample-level outer_val predictions for audited recipes.",
    },
    {
        "file": "outer_fold_selected_features.csv",
        "meaning": "Features selected after refitting prescreening on each outer_train.",
    },
    {
        "file": "outer_fold_scores_rate_matched.csv",
        "meaning": "Outer predictions rescored at final test contact rate with gross value scaled to full test population, without refitting.",
    },
    {
        "file": "outer_config_summary_full_cap.csv",
        "meaning": "Diagnostic aggregation using the original full-cap outer scores.",
    },
    {
        "file": "outer_config_summary.csv",
        "meaning": "Main aggregation using rate-matched, scaled-gross outer scores.",
    },
    {
        "file": "stable_outer_configurations_full_cap.csv",
        "meaning": "Diagnostic finalists from the old full-cap strategy.",
    },
    {
        "file": "stable_outer_configurations.csv",
        "meaning": "Main finalists selected from rate-matched alternative outer evaluation; every row has outer_fold_count equal to the full outer-fold count.",
    },
    {
        "file": "outer_feature_frequency.csv",
        "meaning": "Feature frequency across alternative outer refits.",
    },
    {"file": "stage_status_summary.json", "meaning": "Compact notebook-03 status summary."},
])
output_manifest.to_csv(OUTPUT_FILES["manifest"], index=False)

pd.Series(status_summary)